# HyBrid Embedding $\rightarrow$ LLM

## 1. Bản chất vấn đề

Hiện tại flow của bạn là:

├─ Rule / Extractor

 ├─ Embedding classifier → (category, confidence)
 
 └─ if confidence < threshold → call LLM

👉 LLM đang được dùng như “oracle”
👉 Nhưng nếu không học lại từ kết quả LLM thì:
- Lần sau gặp câu tương tự → vẫn confidence thấp
- → vẫn call LLM
- → vẫn call LLM


→ Chi phí cao + latency cao + không tiến hóa

## 2. Nguyên tắc cải thiện cốt lõi (quan trọng)

Mỗi lần call LLM là một lần bạn mua data label chất lượng cao
→ Phải biến nó thành training signal

Nếu không làm bước này thì pipeline của bạn chết cứng.

## 3. Kỹ thuật cải thiện theo từng level (từ dễ → mạnh)

### LEVEL 1 — Cache & Deduplicate (bắt buộc)
🎯 Mục tiêu
Không gọi LLM cho cùng một intent / semantic giống nhau.
Cách làm
- Lưu embedding + category từ LLM
- So cosine similarity trước khi call LLM

Nếu sim(text, cached_text) > 0.92
→ reuse category
→ skip LLM

In [ ]:
{
  "embedding": [...],
  "category": "Ăn uống",
  "source": "llm"
}

Cái này giảm 30–60% call LLM ngay lập tức

### LEVEL 2 — Auto-expand CATEGORY_EXAMPLES (quan trọng nhất)

Hiện tại bạn có

In [ ]:
CATEGORY_EXAMPLES = {
    "Ăn uống": ["ăn cơm", "đi ăn", "ăn lẩu"],
    ...
}

Khi LLM được gọi → bạn phải làm:

(text, llm_category) → append vào CATEGORY_EXAMPLES


Ví dụ:
"Mua cái gì đó lạ lạ 100k" → "Mua sắm"

➡️ Lần sau:
- Embedding classifier đã có điểm neo mới
- Confidence ↑
- Không gọi LLM nữa

📌 Đây là continual learning không training lại model

### LEVEL 3 — Pseudo-labeling + hard negative mining

Khi nào dùng?
- Sau vài nghìn logs
- Nhiều case confidence thấp

Cách làm:

1. Lưu toàn bộ:

In [ ]:
{
  "text": "...",
  "embedding_category": "...",
  "embedding_confidence": 0.41,
  "llm_category": "Du lịch"
}

2. Offline job:

- Nếu:

LLM category ≠ embedding category
AND confidence < 0.5

→ Đây là hard sample

3. Đưa vào training set / examples

📌 Kỹ thuật này giảm vùng mù của embedding model

### LEVEL 4 — Adaptive threshold (rất hay)

Threshold của bạn đang là fixed:

In [ ]:
CONFIDENCE_THRESHOLD = 0.7

❌ Không tối ưu

#### Cách cải thiện

Threshold theo category

In [ ]:
thresholds = {
  "Ăn uống": 0.6,
  "Y tế": 0.75,
  "Du lịch": 0.8
}

→ Category dễ → threshold thấp

→ Category mơ hồ → threshold cao

📌 Giảm call LLM có kiểm soát

### LEVEL 5 — Train lại embedding classifier (khi đủ data)

Khi bạn có:
- 5k–10k samples
- label từ LLM + user correction

Các hướng:
1. Finetune SimCSE / PhoBERT
2. Train centroid-based classifier
3. Train lightweight head (MLP) trên embedding

📌 Lúc này LLM chỉ còn dùng cho:
- OOD (out-of-distribution)
- category mới

## Tóm tắt:

![](image1.png)